# End-to-End Analytics Pipeline Demo

This notebook walks through the full data-analytics lifecycle using the `analytics` library in this repository. Each section maps to one stage:

1. **Data Loading** &rarr; `dataloading`
2. **Data Cleansing** &rarr; `datacleansing`
3. **Data Exploration (EDA)** &rarr; `dataexploration`
4. **Data Visualization** &rarr; `datavisualization`
5. **Descriptive Analytics** &rarr; `descriptiveanalysis`
6. **Diagnostic Analytics** &rarr; `diagnosticanalysis`
7. **Predictive Analytics** &rarr; `predictiveanalysis`
8. **Prescriptive Analytics** &rarr; `prescriptiveanalysis`

The first part uses a **synthetic sales CSV** (`data/sample_sales.csv`) to demonstrate loading and cleansing of messy real-world-style data. The predictive/ML part also shows a **built-in scikit-learn dataset** so you can see modeling on clean, well-known data.

> **Prerequisite:** run `python data/generate_sample_data.py` once from the project root to create the CSV (see the README).

## 0. Setup

Make the project root importable (so `import analytics` works when running from the `notebooks/` folder) and silence non-critical warnings.

In [ ]:
import sys
import warnings
from pathlib import Path

# Add the project root (parent of notebooks/) to the import path.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from analytics import (
    dataloading,
    datacleansing,
    dataexploration,
    datavisualization as viz,
    descriptiveanalysis,
    diagnosticanalysis,
    predictiveanalysis,
    prescriptiveanalysis,
)

viz.set_style()  # consistent chart styling
print("analytics version:", __import__("analytics").__version__)

## 1. Data Loading

Load the synthetic sales CSV. `dataloading` always returns a standard pandas `DataFrame`, regardless of the source format.

In [ ]:
csv_path = PROJECT_ROOT / "data" / "sample_sales.csv"
raw = dataloading.load_csv(csv_path, parse_dates=["Order Date"])
print("Shape:", raw.shape)
raw.head()

## 2. Data Cleansing

The raw file has messy column names, trailing whitespace, missing values, duplicate rows, and a few extreme outliers. We inspect quality first, then fix it step by step.

In [ ]:
# Quality scorecard BEFORE cleaning.
datacleansing.quality_report(raw)

In [ ]:
# Cleansing steps, each returns a new DataFrame (non-mutating).
df = datacleansing.standardize_column_names(raw)   # 'Order Date' -> 'order_date'
df = datacleansing.strip_whitespace(df)            # trim ' North ' -> 'North'
df = datacleansing.drop_duplicates(df)             # remove duplicate rows
df = datacleansing.handle_missing(df, strategy="median")  # impute numeric NaNs
df = datacleansing.cap_outliers_iqr(df, "sales")   # winsorize extreme sales

print("Clean shape:", df.shape)
print("Remaining missing values:", int(df.isna().sum().sum()))
df.head()

## 3. Data Exploration (EDA)

Understand the dataset's structure, distributions, and relationships before modeling.

In [ ]:
overview = dataexploration.overview(df)
overview

In [ ]:
dataexploration.numeric_summary(df)

In [ ]:
# Strongest pairwise correlations (multicollinearity check).
dataexploration.top_correlations(df, threshold=0.3)

## 4. Data Visualization

Basic and advanced charts. Every plotting function accepts an optional `ax`, so we can compose subplots.

In [ ]:
# Basic charts: distribution + a categorical breakdown.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
viz.plot_histogram(df, "profit", ax=axes[0])
viz.plot_box(df, y="sales", x="region", ax=axes[1])
fig.tight_layout()
plt.show()

In [ ]:
# Advanced chart: correlation heatmap.
viz.plot_correlation_heatmap(df)
plt.show()

## 5. Descriptive Analytics &mdash; "What happened?"

Aggregate the past: sales by region, share of total, and the monthly trend.

In [ ]:
by_region = descriptiveanalysis.share_of_total(df, group_by="region", value="sales")
by_region

In [ ]:
# Monthly sales trend with a 3-period moving average.
monthly = descriptiveanalysis.trend_over_time(
    df, date_column="order_date", value="sales", freq="ME", agg="sum"
)
monthly = descriptiveanalysis.moving_average(monthly, value="sales", window=3)

ax = viz.plot_line(monthly, x="order_date", y="sales")
ax.plot(monthly["order_date"], monthly["sales_ma3"], label="3-mo moving avg", linewidth=2)
ax.legend()
plt.show()

## 6. Diagnostic Analytics &mdash; "Why did it happen?"

Quantify relationships and test whether group differences are statistically meaningful.

> Reminder: these are **associations**, not proof of causation.

In [ ]:
# Which numeric features move most with profit?
diagnosticanalysis.correlation_with_target(df, "profit")

In [ ]:
# Is average profit different between sales channels? (Welch's t-test)
diagnosticanalysis.ttest_two_groups(
    df, group_column="channel", value_column="profit",
    group_a="Online", group_b="Retail",
)

## 7. Predictive Analytics &mdash; "What will happen?"

### 7a. Regression on the sales data
Predict `profit` from the other features. We drop the date column (handled separately in real forecasting) and any direct leakage you want to avoid.

In [ ]:
model_df = df.drop(columns=["order_date", "order_id"])
reg = predictiveanalysis.train_regressor(model_df, target="profit", model="random_forest")
print("Test metrics:", reg["metrics"])

# Feature importances from the fitted model.
importance = predictiveanalysis.get_feature_importance(reg)
viz.plot_feature_importance(importance.values, importance.index, top_n=10)
plt.show()

### 7b. Classification on a built-in scikit-learn dataset
To show the classification path on clean, well-known data we use the **Wine** dataset (predict the cultivar from chemical measurements).

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
wine_df = wine.frame  # features + 'target' column

clf = predictiveanalysis.train_classifier(wine_df, target="target", model="random_forest")
print("Accuracy:", clf["metrics"]["accuracy"])
print("Metrics:", clf["metrics"])
print("Confusion matrix:\n", clf["confusion_matrix"])

## 8. Prescriptive Analytics &mdash; "What should we do?"

Combine business rules and optimization to recommend actions.

In [ ]:
# Rule-based next-best-action on low-margin transactions.
df["margin"] = (df["profit"] / df["sales"]).round(3)
rules = [
    (lambda d: d["margin"] < 0.15, "review_pricing"),
    (lambda d: d["margin"] < 0.25, "monitor"),
]
recommended = prescriptiveanalysis.recommend_by_rules(df, rules, default_action="healthy")
recommended["recommended_action"].value_counts()

In [ ]:
# Optimize a marketing budget across channels by expected per-unit return.
alloc = prescriptiveanalysis.optimize_allocation(
    returns=[0.18, 0.42, 0.10],          # ROI per unit for each channel
    total_budget=100_000,
    max_alloc=[60_000, 60_000, 60_000],  # cap exposure per channel
    labels=["Search", "Social", "Display"],
)
print("Solver status:", alloc.attrs["status"])
alloc

## Summary

We went from a messy raw CSV to cleaned data, explored it, visualized it, summarized it (descriptive), explained it (diagnostic), forecasted it (predictive), and recommended actions (prescriptive) &mdash; all using the reusable `analytics` library. To deploy a model, save the fitted pipeline with `analytics.machinelearning.save_model(reg['pipeline'], 'model.joblib')`.